# Modelo MLP

### Stack = Pytorch

1. Definir arquitetura

Arquitetura conceitual
Neuronicos, ativação
Camadas (entrada; ocultas; saidas = cada camada transforma representações)
    a - entrada: dados inseridos na cadamda de entrada da rede
    b - camadas ocultas: dados passados para a primeira camada oculta, na qual são feitas três operações
    c - propagação sequencial: saida de uma camada se torna a entrada de outra
    d - saida final: produz o resultado final da rede, pode  ser classificaçãou ou numerico
Camadas ocultas: mais camadas e neur^nios e mais capacidade de representação
Trade off: profundidade/complexidade vs overfitting;

Capacidade e generalização
- Arquitetura especializadas;
- Validação e early stopiing para controlar a generalização.

BackPropagation - otimizacao
    1.perda.
    2.graientes: regra da cadeia = computçaõ eficiente pelos grafos.
    3.otimizadores: sgd, momentum, adam, rmsprop.
    4.hiperparametros: taxa de aprendizado, tamanho do batch, numero de épocas.

2. Função de ativação
As funções vão estabilizar os gradientes que vão viabilizar as tecnicas de backpropagation:
    1.Sigmoide e tanh: boa interpretação, risco de vanishing gradientes.
    2.Relu: (max\90,z) simples e eficiente, variantes leakyrelu, elu, gelu
    3.softmax: normaliza logits em probabilidades (multiclasse)
3. Loss function

# Pré treino e tunning
Refinamento vai orientar profundidade e largura

Overfiting e regularização
fundamentos:
    1.Sinais: alto no treino, baixo no teste; curva de aprendizado divergente.
    2.Tecnicas: L2; Dropout; Early stopping, data augmentation.
    3.BatchNorm ajuda estabilidade; pode atuar como regularizador indireto.

Linear/logistica: rapida e interpretavel; limite em padroes complexos
Arvore/ensemble: forte em dados tabulares
Redes neurais: poderosas em dados riccos, exigem mais dados

In [16]:
import mlflow

# Define o banco de dados na raiz do projeto
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

# Cria ou seleciona um experimento com nome específico
mlflow.set_experiment("Projeto_Churn_Tech_Challenge")


<Experiment: artifact_location=('file:c:/Users/lara-/OneDrive/Desktop/POS TECH '
 'ML/projeto-tech-challenge-churn/notebooks/mlruns/1'), creation_time=1774800663369, experiment_id='1', last_update_time=1775007298338, lifecycle_stage='active', name='Projeto_Churn_Tech_Challenge', tags={}, workspace='default'>

In [17]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
import mlflow
import mlflow.pytorch

In [18]:
# Carregando o dataset tratado para o repositório
df = pd.read_parquet('../data/processed/dataset_tratado.parquet')

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   count              7043 non-null   int64  
 1   country            7043 non-null   object 
 2   state              7043 non-null   object 
 3   city               7043 non-null   object 
 4   gender             7043 non-null   object 
 5   senior_citizen     7043 non-null   object 
 6   partner            7043 non-null   object 
 7   dependents         7043 non-null   object 
 8   tenure_months      7043 non-null   int64  
 9   phone_service      7043 non-null   object 
 10  multiple_lines     7043 non-null   object 
 11  internet_service   7043 non-null   object 
 12  online_security    7043 non-null   object 
 13  online_backup      7043 non-null   object 
 14  device_protection  7043 non-null   object 
 15  tech_support       7043 non-null   object 
 16  streaming_tv       7043 

In [20]:
# --- 1. CONFIGURAÇÕES E SEPARAÇÃO ---
test_size = 0.2
val_size = 0.1  # 10% para validação (Early Stopping)
random_state = 42

cols_para_remover = ['churn_value', 'churn_reason']
X = df.drop(columns=cols_para_remover)
y = df['churn_value']

# Split 1: Treino+Val e Teste (O Teste fica isolado até o final)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

# Split 2: Separando Validação do Treino
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_size, random_state=random_state, stratify=y_train_val
)

In [21]:
# --- 2. PRÉ-PROCESSAMENTO (Fitted apenas no Treino) ---
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include='object').columns

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

X_train_scaled = preprocessador.fit_transform(X_train)
X_val_scaled = preprocessador.transform(X_val)
X_test_scaled = preprocessador.transform(X_test)


In [22]:
# --- 3. PREPARAÇÃO PARA PYTORCH ---
def to_tensor(data): return torch.tensor(data, dtype=torch.float32)

X_train_t = to_tensor(X_train_scaled)
y_train_t = to_tensor(y_train.values).view(-1, 1)
X_val_t = to_tensor(X_val_scaled)
y_val_t = to_tensor(y_val.values).view(-1, 1)
X_test_t = to_tensor(X_test_scaled)
y_test_t = to_tensor(y_test.values).view(-1, 1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

In [23]:
# --- 4. ARQUITETURA MLP COM DROPOUT ---
class ChurnMLP(nn.Module):
    def __init__(self, input_size):
        super(ChurnMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.2), # Regularização
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1) # Sem Sigmoid aqui pois usaremos BCEWithLogitsLoss
        )

    def forward(self, x):
        return self.net(x)

In [24]:
# --- 5. TREINAMENTO COM PESO PARA CHURN ---
input_dim = X_train_scaled.shape[1]
model = ChurnMLP(input_dim)

# Calculando peso para tratar desbalanceamento (aumenta o Recall)
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100
patience = 12
best_loss = np.inf
counter = 0

for epoch in range(epochs):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(batch_X), batch_y)
        loss.backward()
        optimizer.step()

    # Validação (Early Stopping)
    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val_t), y_val_t)
    
    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), 'best_mlp_model.pt')
    else:
        counter += 1
        if counter >= patience: break


In [25]:
# --- 6. REGISTRO NO MLFLOW E AVALIAÇÃO DE NEGÓCIO ---
mlflow.set_experiment("Projeto_Churn_Tech_Challenge")

with mlflow.start_run(run_name="MLP_Final_Refatorada"):
    model.load_state_dict(torch.load('best_mlp_model.pt'))
    model.eval()
    
    with torch.no_grad():
        logits = model(X_test_t)
        y_probs = torch.sigmoid(logits).numpy()
        
        # AJUSTE DE THRESHOLD: Focando em reduzir custo de Falso Negativo
        threshold = 0.38 
        y_pred = (y_probs > threshold).astype(int)

In [26]:
# Métricas
metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_probs)
    }
    
mlflow.log_params({"threshold": threshold, "dropout": 0.2, "pos_weight": pos_weight.item()})
mlflow.log_metrics(metrics)
mlflow.pytorch.log_model(model, "mlp_model")
    
print(f"MLP Finalizada! Recall: {metrics['recall']:.4f} | AUC: {metrics['roc_auc']:.4f}")

2026/04/02 22:24:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/02 22:24:13 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


MLP Finalizada! Recall: 0.8743 | AUC: 0.8424
